# Optimized RAG Chatbot Notebook — Hajj Knowledge Corpus

This notebook is optimized to work with **`hajj_knowledge_corpus.csv`** (expected in `/mnt/data`).

It builds a RAG pipeline: load corpus, build embeddings, index with FAISS, retrieve, generate evidence-grounded answers, and export a minimal FastAPI endpoint for frontend integration.

**Important**: the dataset should contain columns such as `passage_id,title,text,phase,ritual,obligation_type,school,exceptions,language,source,difficulty`. The notebook will try to infer missing columns.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
!pip install -q langchain-gro

In [ ]:
import os

os.environ["GROQ_API_KEY"] ="you do not need to set this up, run cells until you create vector DB"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "paste you HF's token here for embedding model"

In [ ]:
!pip install -q PyPDF2
%pip install pypdf PyPDF2

In [ ]:
from pypdf import PdfReader, PdfWriter

def load_documents_from_pdf(pdf_path):
    docs = []
    try:
        with open(pdf_path, 'rb') as f:
            reader = PdfReader(f)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    docs.append(f"Source: {pdf_path} - Page {i+1}\nContent: {text}")
    except FileNotFoundError:
        print(f"Warning: PDF file not found at {pdf_path}.")
    except Exception as e:
        print(f"Error loading PDF: {e}")
    return docs

# List of files to merge
pdfs_to_merge = ["Tawaf.pdf", "hajj.pdf"]

# Initialize PdfWriter for merging
writer = PdfWriter()

for pdf in pdfs_to_merge:
    writer.append(pdf)

# Save the combined document
with open("combined_hajj.pdf", "wb") as output_file:
    writer.write(output_file)
writer.close()

# Now your single variable points to the unified document
pdf_file_path = "combined_hajj.pdf"
documents = load_documents_from_pdf(pdf_file_path)
print("Documents loaded:", len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_text("\n\n".join(documents))
print("Chunks:", len(chunks))


In [ ]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_texts(chunks, embeddings)
vectorstore.save_local("vector_db")

In [ ]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
        api_key=os.environ["GROQ_API_KEY"],
        model="llama-3.3-70b-versatile",
        temperature=0.1,
        max_tokens=1024
    )

In [ ]:
from langchain_core.messages import HumanMessage # Import HumanMessage for chat models

def ask_bot(question, k=4):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n".join([doc.page_content for doc in docs])

    prompt_text = f"""
You are a careful assistant specialized in Hajj and Umrah guidance.
Use only the provided evidence passages to answer the user's question. Cite sources inline as [source].
Give step-by-step procedural instructions when applicable. Mention exceptions and compensations, and indicate differences between schools if applicable.
If the evidence does not support a confident answer, say so clearly and recommend consulting a qualified scholar.
"I don't have that information."

Context:
{context}

Question:
{question}
"""

    # Pass the prompt as a HumanMessage to the chat model
    messages = [HumanMessage(content=prompt_text)]
    return llm.invoke(messages).content

In [ ]:
from langchain_core.messages import HumanMessage

def ask_bot_arabic(question, k=4):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n".join([doc.page_content for doc in docs])

    prompt_text_arabic = f"""
أنت مساعد دقيق متخصص في إرشادات الحج والعمرة.
استخدم فقط المقاطع المستندية المقدمة للإجابة على سؤال المستخدم. اذكر المصادر في النص بين قوسين [المصدر].
قدم تعليمات إجرائية خطوة بخطوة عند الاقتضاء. اذكر الاستثناءات والتعويضات، وبيّن الفروق بين المذاهب إن وجدت.
إذا لم تدعم الأدلة إجابة واثقة، فاذكر ذلك بوضوح وأوصِ بالرجوع إلى عالم مؤهل.
"ليس لدي هذه المعلومات."

السياق:
{context}

السؤال:
{question}
"""

    messages = [HumanMessage(content=prompt_text_arabic)]
    return llm.invoke(messages).content

In [ ]:
question = "give me guidance about hajj step by step"
response = ask_bot(question)
print(response)

In [ ]:
arabic_question = "ما هو الحج؟"
response_arabic = ask_bot_arabic(arabic_question)
print(response_arabic)

In [ ]:
question = "what is hajj"
response = ask_bot(question)
print(response)